# Proliferators vs. Persisters: vem vs. naive, Over Time (Cell + Nucleus)

Same shared embedding as `01_umap_by_condition_over_time.ipynb`, but
compares THREE groups instead of two: naive proliferators, vem
proliferators, and vem persisters (`m_label == config.PROLIF_M_LABEL`
or `config.PERSISTING_M_LABEL`) - showing whether vem's persisters
carve out their own distinct region, rather than just overlapping vem's
own proliferators or naive.

Plots both embeddings `main.py` computes (UMAP and PHATE) every time
this notebook runs, no manual switch needed.

Requires `main.py` to have been run first.


In [ ]:
import os
import sys

import numpy as np
import matplotlib.pyplot as plt

# assumes Jupyter's cwd is this notebook's own directory
# (analyses/01_naive_vs_vem/notebooks/), so the analysis dir - and config.py -
# is one level up, and the repo root - where shared/ lives - is two more
# levels up from there
NOTEBOOK_DIR = os.getcwd()
ANALYSIS_DIR = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..'))
REPO_ROOT    = os.path.abspath(os.path.join(ANALYSIS_DIR, '..', '..'))
sys.path.insert(0, REPO_ROOT)
sys.path.insert(0, ANALYSIS_DIR)
import config

# config.py's OUTPUT_DIR/CAJAL_OUTPUT_DIR are relative to the analysis
# directory, not this notebook's own cwd - re-anchor them to an absolute
# path once, here
config.OUTPUT_DIR       = os.path.join(ANALYSIS_DIR, config.OUTPUT_DIR)
config.CAJAL_OUTPUT_DIR = os.path.join(ANALYSIS_DIR, config.CAJAL_OUTPUT_DIR)

from shared.load_data import load_full_dataset, export_plot_data
from shared.density_plots import export_kde_by_frame


## Plot: embedding with per-group kernel density, proliferators vs. persisters

For each embedding method, one PDF, one SVG per frame, written to
`output/plots/04_prolif_vs_persisters/` (SVGs in their own `svgs/`
subfolder there).


In [ ]:
GROUP_COLORS = {
    'naive_prolif':   'tab:blue',
    'vem_prolif':     'tab:red',
    'vem_persisters': 'tab:purple',
}
NOTEBOOK_PLOTS_DIR = os.path.join(config.OUTPUT_DIR, 'plots', '04_prolif_vs_persisters')
MODALITY_NAMES = {'cell': 'cell', 'nuc': 'nucleus'}

# KDE contour levels (level p encloses the densest (1-p) fraction of a
# group's mass - higher is tighter/closer to the peak) and bandwidth
# multiplier (above 1 smooths over sampling/distance-tie noise)
KDE_LEVELS     = (0.7, 0.8, 0.9)
KDE_BW_ADJUST  = 1

# main.py computes both of these for every modality - plot both every
# time this notebook runs, rather than picking just one
EMBEDDING_METHODS = ['umap', 'phate']


def label_group(df):
    '''
    Restrict to naive proliferators, vem proliferators, and vem
    persisters, and label each with a 'group' column - everything else
    (naive persisters/dying, vem dying, ...) is dropped, since this plot
    only ever compares these three.

    Inputs:
        df: must have: condition, m_label
    Outputs:
        df: subset with a 'group' column added (one of GROUP_COLORS' keys)
    '''
    conditions = [
        (df['condition'] == config.NAIVE_CONDITION) & (df['m_label'] == config.PROLIF_M_LABEL),
        (df['condition'] == config.VEM_CONDITION) & (df['m_label'] == config.PROLIF_M_LABEL),
        (df['condition'] == config.VEM_CONDITION) & (df['m_label'] == config.PERSISTING_M_LABEL),
    ]
    choices = list(GROUP_COLORS.keys())
    df = df.copy()
    df['group'] = np.select(conditions, choices, default=None)
    return df[df['group'].notna()].reset_index(drop=True)


def process_modality(modality):
    '''
    For each embedding method, load this modality's embedding and
    metadata, restrict to naive proliferators / vem proliferators / vem
    persisters, and export a PDF and a per-frame SVG for each
    overlapping frame.

    Inputs:
        modality: 'cell' or 'nuc'
    Outputs:
        group_dfs: {embedding_method: the three-group subset for this
                    modality and embedding method} (in-memory only)
    '''
    group_dfs = {}
    for embedding_method in EMBEDDING_METHODS:
        df = load_full_dataset(config.CAJAL_OUTPUT_DIR, modality, embedding_method)
        group_df = label_group(df)
        print(f"[{modality}/{embedding_method}] {len(group_df)}/{len(df)} cells are in one of the three groups")
        for group in GROUP_COLORS:
            n = (group_df['group'] == group).sum()
            print(f"    {group}: {n} cells")

        export_kde_by_frame(group_df, 'group', GROUP_COLORS, NOTEBOOK_PLOTS_DIR,
                             f'{modality}_{embedding_method}', embedding_method.upper(),
                             title_prefix=f'{MODALITY_NAMES[modality]} morphology (proliferators vs. persisters)',
                             levels=KDE_LEVELS, bw_adjust=KDE_BW_ADJUST)
        group_dfs[embedding_method] = group_df

    return group_dfs


## Run: cell morphology

In [ ]:
cell_group_dfs = process_modality('cell')


## Run: nuclear morphology

In [ ]:
nuc_group_dfs = process_modality('nuc')


## Export plot data

Combines every (modality, embedding method) DataFrame above into one CSV
(`plot_data.csv`, alongside the PDFs/SVGs above) - the same dim_0/dim_1
coordinates and metadata the KDE plots above use, so someone without
this repo can recreate or extend these plots from the raw data alone.

In [ ]:
plot_data = {('cell', m): df for m, df in cell_group_dfs.items()}
plot_data.update({('nuc', m): df for m, df in nuc_group_dfs.items()})
export_plot_data(plot_data, os.path.join(NOTEBOOK_PLOTS_DIR, 'plot_data.csv'))
